In [1]:
!nvidia-smi


Tue May 26 05:45:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
HOME = os.getcwd()
print(HOME)

/kaggle/working


In [3]:
# Pip install method (recommended)

!pip install ultralytics

from IPython import display
display.clear_output()
# prevent ultralytics from tracking your activity
!yolo settings sync=False
import ultralytics
ultralytics.checks()

Ultralytics 8.4.54 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6841.9/8062.4 GB disk)


In [4]:
!pip install ultralytics medpy surface-distance scikit-image -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [5]:
import os
import gc
import cv2
import yaml
import torch
import shutil
import random
import zipfile
import numpy as np
import pandas as pd

from glob import glob
from PIL import Image

from ultralytics import YOLO

from sklearn.model_selection import KFold

from medpy.metric.binary import hd95

from skimage.morphology import skeletonize

from scipy.ndimage import distance_transform_edt

In [6]:
print("CUDA AVAILABLE:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

CUDA AVAILABLE: True
GPU: Tesla T4


In [7]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="pBkm692MeYfeoniNeXHf")
project = rf.workspace("new-workspace-tatne").project("endo-study-3-f9kdm")
version = project.version(2)
dataset = version.download("yolov11")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 116.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0,


Extracting Dataset Version Zip to endo-study-3-2 in yolov11:: 100%|██████████| 3355/3355 [00:00<00:00, 10672.90it/s]


In [8]:
# =========================================================
# DATASET PATH
# =========================================================

SOURCE = "/kaggle/working/endo-study-3-2"

# =========================================================
# OUTPUT PATHS
# =========================================================

ALL_IMG = "/kaggle/working/all/images"

ALL_LBL = "/kaggle/working/all/labels"

GT_MASK_DIR = "/kaggle/working/ground_truth_masks"

FOLD_DIR = "/kaggle/working/10fold"

# =========================================================
# CREATE DIRECTORIES
# =========================================================

os.makedirs(ALL_IMG, exist_ok=True)

os.makedirs(ALL_LBL, exist_ok=True)

os.makedirs(GT_MASK_DIR, exist_ok=True)

os.makedirs(FOLD_DIR, exist_ok=True)

In [9]:
counter = 1

mapping = []

splits = ['train', 'valid', 'test']

for split in splits:

    image_paths = sorted(
        glob(f"{SOURCE}/{split}/images/*")
    )

    for img_path in image_paths:

        ext = os.path.splitext(img_path)[1]

        old_name = os.path.splitext(
            os.path.basename(img_path)
        )[0]

        label_path = (
            f"{SOURCE}/{split}/labels/{old_name}.txt"
        )

        # skip missing labels
        if not os.path.exists(label_path):
            continue

        # skip empty labels
        if os.path.getsize(label_path) == 0:
            continue

        # new sequential name
        new_name = f"img_{counter:06d}"

        new_img_name = new_name + ext

        new_lbl_name = new_name + ".txt"

        # copy image
        shutil.copy(
            img_path,
            os.path.join(ALL_IMG, new_img_name)
        )

        # copy label
        shutil.copy(
            label_path,
            os.path.join(ALL_LBL, new_lbl_name)
        )

        mapping.append([old_name, new_name])

        counter += 1

print("TOTAL IMAGES:", counter - 1)

TOTAL IMAGES: 1645


In [10]:
df = pd.DataFrame(

    mapping,

    columns=["original", "new"]

)

df.to_csv(

    "/kaggle/working/name_mapping.csv",

    index=False

)

print("NAME MAPPING SAVED")

NAME MAPPING SAVED


In [11]:
image_paths = sorted(
    glob(f"{ALL_IMG}/*")
)

for img_path in image_paths:

    img = cv2.imread(img_path)

    h, w = img.shape[:2]

    name = os.path.splitext(
        os.path.basename(img_path)
    )[0]

    label_path = os.path.join(
        ALL_LBL,
        name + ".txt"
    )

    mask = np.zeros((h, w), dtype=np.uint8)

    if os.path.exists(label_path):

        with open(label_path, 'r') as f:

            lines = f.readlines()

        for line in lines:

            data = line.strip().split()

            points = np.array(
                data[1:],
                dtype=np.float32
            )

            points = points.reshape(-1, 2)

            points[:,0] *= w
            points[:,1] *= h

            points = points.astype(np.int32)

            cv2.fillPoly(mask, [points], 255)

    Image.fromarray(mask).save(
        f"{GT_MASK_DIR}/{name}.png"
    )

print("GROUND TRUTH MASKS CREATED")

GROUND TRUTH MASKS CREATED


In [12]:
import os
from glob import glob

LABEL_DIR = "/kaggle/working/all/labels"

empty = 0

for txt in glob(f"{LABEL_DIR}/*.txt"):

    if os.path.getsize(txt) == 0:

        os.remove(txt)

        img = txt.replace("labels", "images")
        img = os.path.splitext(img)[0] + ".jpg"

        if os.path.exists(img):
            os.remove(img)

        empty += 1

print(empty)

0


In [13]:
images = sorted(
    glob(f"{ALL_IMG}/*")
)

random.shuffle(images)

kf = KFold(

    n_splits=10,

    shuffle=True,

    random_state=42

)

for fold, (train_idx, test_idx) in enumerate(kf.split(images)):

    fold_number = fold + 1

    fold_path = f"{FOLD_DIR}/fold_{fold_number}"

    dirs = [

        f"{fold_path}/images/train",

        f"{fold_path}/images/val",

        f"{fold_path}/labels/train",

        f"{fold_path}/labels/val"

    ]

    for d in dirs:

        os.makedirs(d, exist_ok=True)

    # TRAIN
    for idx in train_idx:

        img_path = images[idx]

        name = os.path.basename(img_path)

        label_name = os.path.splitext(name)[0] + ".txt"

        shutil.copy(

            img_path,

            f"{fold_path}/images/train/{name}"

        )

        shutil.copy(

            f"{ALL_LBL}/{label_name}",

            f"{fold_path}/labels/train/{label_name}"

        )

    # VAL
    for idx in test_idx:

        img_path = images[idx]

        name = os.path.basename(img_path)

        label_name = os.path.splitext(name)[0] + ".txt"

        shutil.copy(

            img_path,

            f"{fold_path}/images/val/{name}"

        )

        shutil.copy(

            f"{ALL_LBL}/{label_name}",

            f"{fold_path}/labels/val/{label_name}"

        )

    # YAML
    yaml_data = {

        "path": fold_path,

        "train": "images/train",

        "val": "images/val",

        "names": {

            0: "tooth"

        }

    }

    with open(f"{fold_path}/data.yaml", "w") as f:

        yaml.dump(yaml_data, f)

print("10 FOLDS CREATED")

10 FOLDS CREATED


In [14]:
def dice_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    intersection = np.logical_and(gt, pred).sum()

    return (
        2.0 * intersection
    ) / (
        gt.sum() + pred.sum() + 1e-8
    )

# =========================================================

def iou_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    intersection = np.logical_and(gt, pred).sum()

    union = np.logical_or(gt, pred).sum()

    return intersection / (union + 1e-8)

# =========================================================

def precision_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    tp = np.logical_and(gt, pred).sum()

    fp = np.logical_and(~gt, pred).sum()

    return tp / (tp + fp + 1e-8)

# =========================================================

def recall_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    tp = np.logical_and(gt, pred).sum()

    fn = np.logical_and(gt, ~pred).sum()

    return tp / (tp + fn + 1e-8)

# =========================================================

def cldice(gt, pred):

    gt = gt > 0
    pred = pred > 0

    skel_gt = skeletonize(gt)

    skel_pred = skeletonize(pred)

    tprec = (
        np.logical_and(
            skel_pred,
            gt
        ).sum()
    ) / (skel_pred.sum() + 1e-8)

    tsens = (
        np.logical_and(
            skel_gt,
            pred
        ).sum()
    ) / (skel_gt.sum() + 1e-8)

    return (
        2 * tprec * tsens
    ) / (tprec + tsens + 1e-8)

# =========================================================

def hd95_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    if gt.sum() == 0 or pred.sum() == 0:
        return np.nan

    return hd95(pred, gt)

# =========================================================

def assd_score(gt, pred):

    gt = gt > 0
    pred = pred > 0

    if gt.sum() == 0 or pred.sum() == 0:
        return np.nan

    dt_gt = distance_transform_edt(~gt)

    dt_pred = distance_transform_edt(~pred)

    sds_gt = dt_pred[gt]

    sds_pred = dt_gt[pred]

    return (
        sds_gt.mean() + sds_pred.mean()
    ) / 2

In [15]:
MODEL = "yolo11s-seg.pt"

IMG_SIZE = 640

BATCH = 8

EPOCHS = 200

PATIENCE = 40

# =========================================================
# WHICH FOLDS TO RUN
# =========================================================

START_FOLD = 8
END_FOLD = 10

In [16]:
for fold in range(START_FOLD, END_FOLD + 1):

    print("\n")
    print("="*60)
    print(f"TRAINING FOLD {fold}")
    print("="*60)

    fold_path = f"{FOLD_DIR}/fold_{fold}"

    yaml_path = f"{fold_path}/data.yaml"

    # =====================================================
    # LOAD MODEL
    # =====================================================

    model = YOLO(MODEL)

    # =====================================================
    # TRAIN
    # =====================================================

    model.train(

        data=yaml_path,

        epochs=EPOCHS,

        patience=PATIENCE,

        imgsz=IMG_SIZE,

        batch=BATCH,

        device=0,

        workers=2,

        optimizer="AdamW",

        lr0=0.002,

        cache=False,

        amp=True,

        deterministic=False,

        overlap_mask=False,

        close_mosaic=10,

        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,

        degrees=5,
        translate=0.05,
        scale=0.2,

        fliplr=0.5,

        project=fold_path,

        name="train",

        task="segment",

        save=True,

        plots=False

    )

    # =====================================================
    # LOAD BEST MODEL
    # =====================================================

    best_model = (
        f"{fold_path}/train/weights/best.pt"
    )

    model = YOLO(best_model)

    # =====================================================
    # TEST IMAGES
    # =====================================================

    test_images = sorted(
        glob(f"{fold_path}/images/val/*")
    )

    # =====================================================
    # PREDICT
    # =====================================================

    results = model.predict(

        source=test_images,

        imgsz=IMG_SIZE,

        conf=0.25,

        retina_masks=True,

        save=False,

        verbose=False

    )

    # =====================================================
    # SAVE PREDICTED MASKS
    # =====================================================

    pred_mask_dir = (
        f"{fold_path}/predicted_masks"
    )

    os.makedirs(pred_mask_dir, exist_ok=True)

    for result in results:

        img_name = os.path.basename(
            result.path
        )

        stem = os.path.splitext(
            img_name
        )[0]

        orig_shape = result.orig_shape

        h, w = orig_shape

        combined_mask = np.zeros(
            (h, w),
            dtype=np.uint8
        )

        # =================================================
        # NO PREDICTION
        # =================================================

        if result.masks is None:

            Image.fromarray(
                combined_mask
            ).save(
                f"{pred_mask_dir}/{stem}.png"
            )

            continue

        # =================================================
        # GET MASKS
        # =================================================

        masks = result.masks.data.cpu().numpy()

        for mask in masks:

            mask = cv2.resize(

                mask,

                (w, h),

                interpolation=cv2.INTER_NEAREST

            )

            binary = (
                (mask > 0.5)
                .astype(np.uint8)
                * 255
            )

            combined_mask = np.maximum(
                combined_mask,
                binary
            )

        Image.fromarray(
            combined_mask
        ).save(
            f"{pred_mask_dir}/{stem}.png"
        )

    print(f"PREDICTED MASKS SAVED FOR FOLD {fold}")

    # =====================================================
    # EVALUATION
    # =====================================================

    dice_list = []
    iou_list = []
    precision_list = []
    recall_list = []
    cldice_list = []
    hd95_list = []
    assd_list = []

    pred_masks = sorted(
        glob(f"{pred_mask_dir}/*.png")
    )

    for pred_path in pred_masks:

        name = os.path.basename(pred_path)

        gt_path = os.path.join(
            GT_MASK_DIR,
            name
        )

        if not os.path.exists(gt_path):
            continue

        pred = cv2.imread(pred_path, 0)

        gt = cv2.imread(gt_path, 0)

        pred = pred > 127

        gt = gt > 127

        # =================================================
        # METRICS
        # =================================================

        dice_list.append(
            dice_score(gt, pred)
        )

        iou_list.append(
            iou_score(gt, pred)
        )

        precision_list.append(
            precision_score(gt, pred)
        )

        recall_list.append(
            recall_score(gt, pred)
        )

        cldice_list.append(
            cldice(gt, pred)
        )

        hd95_list.append(
            hd95_score(gt, pred)
        )

        assd_list.append(
            assd_score(gt, pred)
        )

    # =====================================================
    # SAVE CSV
    # =====================================================

    results_dict = {

        "fold": fold,

        "dice": np.nanmean(dice_list),

        "iou": np.nanmean(iou_list),

        "precision": np.nanmean(precision_list),

        "recall": np.nanmean(recall_list),

        "cldice": np.nanmean(cldice_list),

        "hd95": np.nanmean(hd95_list),

        "assd": np.nanmean(assd_list)

    }

    metrics_csv = (
        "/kaggle/working/fold_results8_1.csv"
    )

    df = pd.DataFrame([results_dict])

    if os.path.exists(metrics_csv):

        df.to_csv(

            metrics_csv,

            mode='a',

            header=False,

            index=False

        )

    else:

        df.to_csv(

            metrics_csv,

            index=False

        )

    print("METRICS SAVED")

    # =====================================================
    # ZIP FOLD
    # =====================================================

    shutil.make_archive(

        fold_path,

        'zip',

        fold_path

    )

    print(f"FOLD {fold} ZIPPED")

    # =====================================================
    # CLEAR MEMORY
    # =====================================================

    del model

    gc.collect()

    torch.cuda.empty_cache()

    print(f"FOLD {fold} COMPLETED")

print("\nALL FOLDS COMPLETED")



TRAINING FOLD 8
Ultralytics 8.4.54 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/10fold/fold_8/data.yaml, degrees=5, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, ov

Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice


FOLD 8 ZIPPED
FOLD 8 COMPLETED


TRAINING FOLD 9
Ultralytics 8.4.54 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/10fold/fold_9/data.yaml, degrees=5, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, opti

Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice


FOLD 9 ZIPPED
FOLD 9 COMPLETED


TRAINING FOLD 10
Ultralytics 8.4.54 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/10fold/fold_10/data.yaml, degrees=5, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, op

Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice
Mean of empty slice


FOLD 10 ZIPPED
FOLD 10 COMPLETED

ALL FOLDS COMPLETED


In [17]:
df = pd.read_csv(
    "/kaggle/working/fold_results8_1.csv"
)

metrics = [

    "dice",

    "iou",

    "precision",

    "recall",

    "cldice",

    "hd95",

    "assd"

]

print("\nFINAL RESULTS\n")

for metric in metrics:

    mean = df[metric].mean()

    std = df[metric].std()

    print(

        f"{metric}: "

        f"{mean:.4f} ± {std:.4f}"

    )


FINAL RESULTS

dice: nan ± nan
iou: nan ± nan
precision: nan ± nan
recall: nan ± nan
cldice: nan ± nan
hd95: nan ± nan
assd: nan ± nan
